# Prepare Electron Microscopy Data

The powerfit program requires a EM density of a unknown structure where it can fit structures into.

Most of the the EM density contains multiple structures, so we want to remove all but the unknown structure.

For this example we will use [EMD-33292](https://www.ebi.ac.uk/emdb/EMD-33292), a sodium channel, with fitted model [7xm9](https://www.ebi.ac.uk/pdbe/entry/pdb/7xm9).

The fitted model consist of following chains:
- A: Isoform 3 of Sodium channel protein type 9 subunit alpha,Green fluorescent protein
- B: Sodium channel subunit beta-1,Green fluorescent protein
- C: Sodium channel subunit beta-2

We will use the B chain, the sodium channel subunit beta-1, as the unknown structure.

Lets start by downloading the density map and the fitted model.

In [1]:
!wget -nc https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-33292/map/emd_33292.map.gz
!gunzip -kf emd_33292.map.gz
!wget -nc https://www.ebi.ac.uk/pdbe/entry-files/download/7xm9.cif

File ‘emd_33292.map.gz’ already there; not retrieving.

File ‘7xm9.cif’ already there; not retrieving.



In [2]:
# TODO visualize with molviewspec in Mol*

## Prepare density uing Chimerax

Let us use [ChimeraX](https://www.rbvi.ucsf.edu/chimerax/) to prepare the EM density.


Write a ChimeraX command script (.cxc) to mask everything but the B chain in the denstiy map from EMDB.

In [3]:
from pathlib import Path

in_density = Path("emd_33292.map")
pdb = Path("7xm9.cif")
unknown_chain = "B"
resolution = 3.48
masked_density = Path(f"{in_density.stem}-{pdb.stem}-{unknown_chain}-{resolution}.mrc")
script = masked_density.with_suffix(".cxc")
script.write_text(f"""\
open {in_density};
open {pdb};
delete #2/{unknown_chain};
molmap #2 {resolution};
volume mask #1 surfaces #3 invertMask true;
save {masked_density} #4;
exit
""")

148

In [18]:
!chimerax --nogui --script $script

Executing: runscript emd_33292-7xm9-B-3.48.cxc
Executing: open emd_33292.map
Computing emd_33292.map surface, level 0.293
Calculated emd_33292.map surface, level 0.293, with 805008 triangles
Opened emd_33292.map as #1, grid size 256,256,256, pixel 1.04, shown at level 0.293, step 1, values float32
Executing: open 7xm9.cif
_7xm9.cif_ title:  
**Cryo-EM structure of human NaV1.7/beta1/beta2-XEN907**
[[more info...]](cxcmd:log metadata #2)  
  

Chain information for 7xm9.cif #2  
---  
Chain | Description | UniProt  
[A](cxcmd:select /A:7-1769 "Select chain") | [Isoform 3 of Sodium channel protein type 9 subunit alpha,Green fluorescent protein](cxcmd:sequence chain #2/A "Show sequence") | [SCN9A_HUMAN](cxcmd:open Q15858 from uniprot associate #2/A "Show annotations") [1-1988](cxcmd:select #2/A:1-1988 "Select sequence")  
[B](cxcmd:select /B:20-192 "Select chain") | [Sodium channel subunit beta-1,Green fluorescent protein](cxcmd:sequence chain #2/B "Show sequence") | [SCN1B_HUMAN](cxcmd:o

In [12]:
# Clean after ourselves
script.unlink()

In [10]:
masked_density, pdb

(PosixPath('emd_33292-7xm9-B-3.48.mrc'), PosixPath('7xm9.cif'))

Open the `masked_density` file in a viewer to verify that densities of chain A and C from `pdb` file have been removed.

To check that the created density can be used by powerfit we can run powerfit with it.

## Create a re-oriented template structure

We could just fit chain B from 7xm9.cif, but as it is already in the correct position and orientation this feels a bit like cheating.
we can make a new template structure that has incorrect position and orientation.

Let us use the [atomium](https://atomium.bio/) to apply a rotation and translation to the B chain of 7xm9.cif.


In [20]:
import atomium

p = atomium.open(str(pdb))
chain = p.model.chain(unknown_chain)
chain.rotate(0.5, "x")
chain.rotate(0.1, "y")
chain.rotate(0.2, "z")
chain.translate(40, 50, 60)
template = pdb.with_suffix(".B-reoriented.pdb")
chain.save(str(template))

## Fit with powerfit

In [11]:
powerfit_result_dir = Path(f"powerfit-{masked_density.stem}")

In [12]:
!powerfit $masked_density $resolution $template -d $powerfit_result_dir --laplace --delimiter , -p 1

Target file read from:                                                          
/home/verhoes/git/protein-detective/protein-detective/docs/emd_33292-7xm9-B-3.48
.mrc                                                                            
Target resolution: 3.48                                                         
Initial shape of density: 256 256 256                                           
Shape after trimming: 175 163 149                                               
Shape after extending: 175 168 150                                              
Template file read from:                                                        
/home/verhoes/git/protein-detective/protein-detective/docs/7xm9.B-reoriented.cif
Reading in rotations.                                                           
Requested rotational sampling density: 10.00                                    
Real rotational sampling density: 9.72                                          
Requested number of processo

In [13]:
powerfit_results = powerfit_result_dir / "solutions.out"

In [16]:
!head $powerfit_results

rank,cc,Fish-z,rel-z,x,y,z,a11,a12,a13,a21,a22,a23,a31,a32,a33
1,0.261,0.267,17.728,123.760,101.920,169.520,0.978,0.176,-0.109,-0.109,0.886,0.450,0.176,-0.429,0.886
2,0.211,0.214,14.221,122.720,101.920,168.480,0.978,0.176,-0.109,-0.109,0.886,0.450,0.176,-0.429,0.886
3,0.175,0.176,11.711,123.760,102.960,168.480,0.978,0.176,-0.109,-0.109,0.886,0.450,0.176,-0.429,0.886


## Fit with Chimerax

Chimerax has a [fitmap command](https://www.rbvi.ucsf.edu/chimerax/docs/user/commands/fitmap.html) that can fit a structure into a density map.

In [17]:
fit_script = masked_density.with_suffix(".fit.cxc")
fits = masked_density.with_suffix(".chimerax.fits")
fit_script.write_text(f"""\
open {template};
open {masked_density}
fitmap #1 inMap #2 logFits {fits};
exit
""")

128

In [18]:
!chimerax --nogui --script $fit_script

Executing: runscript emd_33292-7xm9-B-3.48.fit.cxc
Executing: open 7xm9.B-reoriented.cif
Summary of feedback from opening 7xm9.B-reoriented.cif  
---  
_warnings_ | Unknown polymer entity '1' on line 13  
Missing entity information. Treating each chain as a separate entity.  
Missing or incorrect sequence information. Inferred polymer connectivity.  
  
  

Chain information for 7xm9.B-reoriented.cif #1  
---  
Chain | Description  
[B](cxcmd:select /B:20-192 "Select chain") | [No description available](cxcmd:sequence chain #1/B "Show sequence")  
  

Computing secondary structure
Executing: open emd_33292-7xm9-B-3.48.mrc
Computing emd_33292-7xm9-B-3.48.mrc surface, level 0.253
Calculated emd_33292-7xm9-B-3.48.mrc surface, level 0.253, with 892784 triangles
Opened emd_33292-7xm9-B-3.48.mrc as #2, grid size 256,256,256, pixel 1.04, shown at level 0.253, step 1, values float32
Executing: fitmap #1 inMap #2 logFits emd_33292-7xm9-B-3.48.chimerax.fits
Fit molecule 7xm9.B-reoriented.cif (#1

In [19]:
print(fits.read_text())

Rxx Rxy Rxz Ryx Ryy Ryz Rzx Rzy Rzz Tx Ty Tz correlation correlation_about_mean overlap average_map_value points atoms_outside_contour clash contour_level steps shift angle
0.98918 -0.12499 -0.076816 0.12029 0.99074 -0.062971 0.083975 0.053049 0.99505 25.703 -7.6938 -16.141 None None None -0.00018782 1416 1416 None 0.25348 1200 6.6375 9.0735
